# Helix first turn, core and last turn: SLAE ¹⁵N error (PANAV, 10-seed ensemble)

First turn = Ncap, N1–N3; core = N4–C4; last turn = C3–C1, Ccap (Richardson & Richardson 1988; Aurora & Rose 1998), from α-helices of ≥ 6 residues. Residues: GT present, prediction mask True, no proline.

Statistics: protein-level Wilcoxon signed-rank test (Holm-adjusted), residue-level Mann-Whitney U test, protein-cluster bootstrap CI.

In [ ]:
import json
import os
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, wilcoxon

PRED_PATH = ("../../../data/analysis/Fig4/results/predictions_N_seed_ensemble_10seed.json")
GT_PATH = ("../../../data/rBMRB/cluster_30p/PANAV/test.json")
STRUCT_DIR = Path("../../../data/rBMRB/cluster_30p/structures_split_filtered/test")
REC_PATH = ("../../../data/analysis/Fig4/sasa_results/per_residue_records.json")     # exposure (buried / exposed)
DSSP_CACHE = Path("../../../data/analysis/Fig4/dssp_results/dssp8_test_filtered.json")
# If mkdssp fails with 'Could not load dictionary mmcif_pdbx.dic', point libcifpp to its data, e.g.
#   export LIBCIFPP_DATA_DIR=$CONDA_PREFIX/share/libcifpp
MKDSSP_BIN = "mkdssp"
RECORDS_PATH = ("../../../data/analysis/Fig4/all_features/per_residue_records_full.json")
SS_SOURCE = "dssp8"       # "dssp8": DSSP H only (alpha-helix); "records": ss_class of per_residue_records_full (H+G+I)
FIG_DIR = Path(os.environ.get("HELIXCAP_FIG_DIR", "Figs"))    # set HELIXCAP_FIG_DIR for a dry run elsewhere
FIG_NAME = "helix_termini_error_panav10seedens"     # stats table
FIG_DIR.mkdir(parents=True, exist_ok=True)

BIG_ERR = 3.0      # ppm, "large error"
MIN_HELIX = 6      # helices shorter than this get no N1-N3 / C3-C1 labels
N_BOOT = 2000


def set_style(base=12, family="Arial"):
    plt.rcParams.update({
        "font.family": family, "font.size": base,
        "axes.titlesize": base + 2, "axes.labelsize": base,
        "xtick.labelsize": base - 2, "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2, "svg.fonttype": "none",
    })

## 1 · Secondary structure per residue


In [ ]:
def run_dssp(pdb_path):
    # {resnum: DSSP 8-state code} for one structure ('-' for coil)
    import warnings
    from Bio.PDB import PDBParser
    from Bio.PDB.DSSP import DSSP
    warnings.filterwarnings("ignore")
    model = PDBParser(QUIET=True).get_structure("m", pdb_path)[0]
    dssp = DSSP(model, str(pdb_path), file_type="PDB", dssp=MKDSSP_BIN)
    return Path(pdb_path).stem, {int(k[1][1]): dssp[k][2] for k in dssp.keys()}


if SS_SOURCE == "records":
    # 'H' for every residue whose ss_class is helix, '-' otherwise
    dssp8 = {e: {r["resnum"]: ("H" if r["ss_class"] == "helix" else "-") for r in recs}
             for e, recs in json.load(open(RECORDS_PATH)).items()}
elif DSSP_CACHE.exists():
    dssp8 = {e: {int(r): c for r, c in d.items()} for e, d in json.load(open(DSSP_CACHE)).items()}
else:
    with ThreadPoolExecutor(max_workers=8) as pool:      # mkdssp is an external program, threads suffice
        dssp8 = dict(pool.map(run_dssp, sorted(STRUCT_DIR.glob("*.pdb"))))
    json.dump(dssp8, open(DSSP_CACHE, "w"))
print(f"secondary structure for {len(dssp8)} structures, source = {SS_SOURCE}")

## 2 · Helix positions (Richardson nomenclature)

In [ ]:
def helix_positions(codes):
    # {resnum: position label} from the DSSP codes of one protein
    resnums = sorted(codes)
    helices, run = [], []
    for r in resnums:                                   # maximal runs of 'H' with consecutive numbering
        if codes[r] == "H" and (not run or r == run[-1] + 1):
            run.append(r)
        else:
            if run:
                helices.append(run)
            run = [r] if codes[r] == "H" else []
    if run:
        helices.append(run)

    label = {}
    for h in helices:
        for r in h:
            label[r] = "Mid"
        if len(h) >= MIN_HELIX:
            for k, name in enumerate(["N1", "N2", "N3"]):
                label[h[k]] = name
            for k, name in enumerate(["C1", "C2", "C3"]):
                label[h[-1 - k]] = name
        else:
            for r in h:
                label[r] = "short helix"
    for h in helices:                                   # caps and flanks never overwrite a helical residue
        long_helix = len(h) >= MIN_HELIX
        for r, name in [(h[0] - 1, "Ncap"), (h[-1] + 1, "Ccap")]:
            if r in codes and r not in label:
                label[r] = name if long_helix else "short-helix cap"
        for r, name in [(h[0] - 2, "N'"), (h[-1] + 2, "C'")]:
            if r in codes and r not in label:
                label[r] = name
    return label, len(helices)


pred = json.load(open(PRED_PATH))
gt = json.load(open(GT_PATH))
key_of = {v: k for k, v in gt["entry_ID"].items()}
exposure = {e: {r["resnum"]: r["exposure"] for r in recs} for e, recs in json.load(open(REC_PATH)).items()}

rows, n_helices = [], 0
for entry in pred["mean"]:
    if entry not in key_of or entry not in dssp8:
        continue
    k = key_of[entry]
    seq, exp_n = gt["sequence"][k], gt["N"][k]
    mean, mask = pred["mean"][entry], pred["mask"][entry]
    label, nh = helix_positions(dssp8[entry])
    n_helices += nh
    for i, aa in enumerate(seq):
        r = i + 1
        if exp_n[i] is None or aa == "P" or not mask[i] or mean[i] is None:    # standard gate
            continue
        rows.append(dict(entry=entry, resnum=r, aa=aa, dssp=dssp8[entry].get(r), position=label.get(r, "other"),
                         exposure=exposure.get(entry, {}).get(r), err=abs(mean[i] - exp_n[i])))
df = pd.DataFrame(rows)
# the three groups along a helix (position labels from helix_positions)
GROUPS = {
    "first turn": dict(positions=["Ncap", "N1", "N2", "N3"], label="First turn\n(Ncap–N3)"),
    "core": dict(positions=["Mid"], label="Core\n(N4–C4)"),
    "last turn": dict(positions=["C3", "C2", "C1", "Ccap"], label="Last turn\n(C3–Ccap)"),
}
PAIRS = [("core", "first turn"), ("core", "last turn"), ("last turn", "first turn")]   # (reference, tested)
print(f"{len(df)} gated residues, {df.entry.nunique()} proteins, {n_helices} helices")
print(df.position.value_counts().to_string())

## 3 · First turn, core and last turn: pairwise comparisons


In [ ]:
def protein_bootstrap(sub, a, b, n_boot=N_BOOT, seed=0):
    # median(b) - median(a) on the 'group' column, proteins resampled with replacement
    rng = np.random.default_rng(seed)
    ent, val, grp = sub.entry.to_numpy(), sub.err.to_numpy(), sub.group.to_numpy()
    idx_of = {e: np.flatnonzero(ent == e) for e in np.unique(ent)}
    proteins = np.array(list(idx_of))
    diffs = []
    for _ in range(n_boot):
        idx = np.concatenate([idx_of[p] for p in rng.choice(proteins, proteins.size)])
        va, vb = val[idx][grp[idx] == a], val[idx][grp[idx] == b]
        if len(va) >= 5 and len(vb) >= 5:
            diffs.append(np.median(vb) - np.median(va))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return np.median(val[grp == b]) - np.median(val[grp == a]), lo, hi


def compare(ref_positions, test_positions):
    # one row of statistics: tested group vs reference group
    sub = df[df.position.isin(test_positions + ref_positions)].copy()
    sub["group"] = np.where(sub.position.isin(test_positions), "test", "ref")
    test, ref = sub[sub.group == "test"].err, sub[sub.group == "ref"].err
    delta, lo, hi = protein_bootstrap(sub, "ref", "test")
    # protein level: per-protein difference of medians, Wilcoxon signed-rank test against 0
    per_protein = sub.groupby(["entry", "group"]).err.median().unstack().dropna()
    diff = per_protein["test"] - per_protein["ref"]
    return dict(n_test=len(test), n_ref=len(ref), median_test=test.median(), median_ref=ref.median(),
                delta_median=delta, ci_lo=lo, ci_hi=hi,
                big_test=(test >= BIG_ERR).mean(), big_ref=(ref >= BIG_ERR).mean(),
                p_mwu=mannwhitneyu(test, ref, alternative="two-sided").pvalue,
                n_proteins=len(diff), proteins_test_higher=int((diff > 0).sum()),
                p_wilcoxon_protein=wilcoxon(diff).pvalue)


def holm(p):
    # Holm step-down adjustment
    p = np.asarray(p, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(p) - rank) * p[i])
        adj[i] = min(running, 1.0)
    return adj


rows = [dict(comparison=f"{test} vs {ref}", ref=ref, test=test,
             **compare(GROUPS[ref]["positions"], GROUPS[test]["positions"])) for ref, test in PAIRS]
stats = pd.DataFrame(rows).set_index("comparison")
stats["p_mwu_holm"] = holm(stats.p_mwu)
stats["p_wilcoxon_protein_holm"] = holm(stats.p_wilcoxon_protein)
pd.set_option("display.width", 250)
pd.set_option("display.float_format", lambda v: f"{v:.3g}")
print(stats.T.to_string())

# each terminal position against the core
core = df[df.position.isin(GROUPS["core"]["positions"])].err
print(f"\nposition vs core (median {core.median():.2f} ppm, large-error rate {(core >= BIG_ERR).mean():.3f})")
for p in GROUPS["first turn"]["positions"] + GROUPS["last turn"]["positions"]:
    c = df[df.position == p].err
    print(f"  {p:4s}: n {len(c):4d}, median {c.median():.2f} ppm, large-error rate {(c >= BIG_ERR).mean():.3f}, "
          f"MWU P = {mannwhitneyu(c, core, alternative='two-sided').pvalue:.2g}")
stats.to_csv(FIG_DIR / f"{FIG_NAME}_stats.csv")


## 4 · Figure

In [ ]:
set_style(13)
COL = {"first turn": "#C678DD", "core": "#E06C75", "last turn": "#61AFEF"}
ORDER = ["first turn", "core", "last turn"]
XPOS = {g: i + 1 for i, g in enumerate(ORDER)}


def sci(p):
    # P value in the publication format, e.g. 4.3 x 10^-24
    if p >= 1e-3:
        return f"P = {p:.2g}"
    m, e = f"{p:.1e}".split("e")
    return rf"P = {m} $\mathregular{{\times\ 10^{{{int(e)}}}}}$"


def stars(p):
    # significance stars: **** P < 1e-4, *** < 1e-3, ** < 0.01, * < 0.05, ns otherwise
    for cut, s in [(1e-4, "****"), (1e-3, "***"), (1e-2, "**"), (5e-2, "*")]:
        if p < cut:
            return s
    return "ns"


def whisker_top(values):
    q1, q3 = np.percentile(values, [25, 75])
    return q3 + 1.5 * (q3 - q1)


data = [df[df.position.isin(GROUPS[g]["positions"])].err.values for g in ORDER]
fig, ax = plt.subplots(figsize=(5.2, 5.6))
b = ax.boxplot(data, positions=[XPOS[g] for g in ORDER], patch_artist=True, showfliers=False, widths=0.6,
               medianprops=dict(color="black", lw=1.2))
for box, g in zip(b["boxes"], ORDER):
    box.set(facecolor=COL[g], alpha=0.85, edgecolor="black", linewidth=0.8)

# brackets: adjacent pairs first, the long first-vs-last bracket on top
y0 = max(whisker_top(v) for v in data) * 1.04
step = y0 * 0.22
levels = {("core", "first turn"): 0, ("core", "last turn"): 0, ("last turn", "first turn"): 1}
for (ref, test), level in levels.items():
    r = stats.loc[f"{test} vs {ref}"]
    x1, x2 = sorted([XPOS[ref], XPOS[test]])
    x1, x2 = x1 + 0.04, x2 - 0.04                      # small gap so adjacent brackets do not touch
    y = y0 + level * step
    ax.plot([x1, x1, x2, x2], [y, y + step * 0.08, y + step * 0.08, y], color="black", lw=1)
    ax.text((x1 + x2) / 2, y + step * 0.16,
            f"{stars(r.p_wilcoxon_protein_holm)}\nΔ {r.delta_median:+.2f} ppm",
            ha="center", va="bottom", fontsize=10, font="Arial")

ax.set_xticks([XPOS[g] for g in ORDER])
ax.set_xticklabels([f"{GROUPS[g]['label']}\nn = {len(v)}" for g, v in zip(ORDER, data)])
ax.set_ylabel(r"$\mathregular{^{15}}$N |error| (ppm)")
ax.set_ylim(-0.12, y0 + 2.2 * step)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.tight_layout()
fig.savefig(FIG_DIR / f"{FIG_NAME}.svg", bbox_inches="tight")
fig.savefig(FIG_DIR / f"{FIG_NAME}.png", dpi=300, bbox_inches="tight")
print("saved", FIG_DIR / f"{FIG_NAME}.svg")
plt.show()
